# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de aplicar `Flatten()`, cada imagen tendrá **32 × 32 = 1.024 valores**, uno por cada píxel en escala de grises.

## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())

In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()

In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** `Flatten()` convierte la matriz de píxeles de 32 × 32 en un vector de 1.024 valores. La capa `Dense` recibe un vector de características por ejemplo.

**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:** Porque CIFAR-10 tiene 10 clases. Cada neurona produce un logit asociado a una de ellas.

**4.** ¿Qué función cumple `ReLU`?

**Respuesta:** ReLU introduce no linealidad y devuelve `max(0, x)`: los valores negativos se convierten en cero y los positivos se mantienen. Esto permite aprender relaciones más complejas que una transformación lineal.

---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:** Porque la última capa entrega logits (valores sin convertir en probabilidades), ya que no tiene Softmax. Esta opción indica a la función de pérdida que debe interpretarlos correctamente y aplicar internamente el cálculo numéricamente estable correspondiente.

---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)

### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:** Una época es una pasada completa por todos los ejemplos de entrenamiento.

**7.** ¿Qué representa `batch_size=64`?

**Respuesta:** El modelo procesa 64 ejemplos antes de actualizar sus pesos. Como hay 50.000 imágenes de entrenamiento, cada época requiere aproximadamente 782 lotes.

## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:** Hay indicios de overfitting si la precisión de entrenamiento sigue aumentando mientras la precisión de validación se estanca o disminuye y la diferencia entre ambas crece. Si las dos curvas se mantienen cercanas, no hay evidencia fuerte de overfitting. Confirma cuál de estos patrones aparece en las curvas generadas al ejecutar el entrenamiento.

---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")

### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:** En general, CIFAR-10 en escala de grises es más difícil para esta red que Fashion-MNIST. Aunque ambos tienen imágenes de 28×28 o 32×32 y 10 clases, CIFAR-10 contiene objetos y escenas más variados, y convertirlos a gris elimina información de color. Al ejecutar el notebook, compara el `Test accuracy` impreso con el accuracy que obtuviste en Fashion-MNIST: un valor menor respalda esta conclusión.

---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits, axis=1)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])

In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:** Softmax transforma los logits en valores no negativos normalizados: divide la exponencial de cada logit entre la suma de las exponenciales de todos los logits. Por eso, las probabilidades de las 10 clases suman 1, salvo pequeñas diferencias de redondeo.

---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta:** Es el valor de `Test accuracy` que imprime la celda de evaluación (también se muestra como porcentaje). Debe completarse con el resultado obtenido al ejecutar el notebook; puede variar según el entorno y el entrenamiento.

### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:** Es común que clases visualmente parecidas, como `cat` y `dog`, o algunas clases de animales, se confundan. La dificultad concreta se debe verificar con las predicciones y las curvas generadas; no todas las ejecuciones tienen necesariamente los mismos errores.

### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante sobre la imagen podría perderse al tratarla únicamente como un vector?

**Respuesta:** Se pierde la representación explícita de la estructura espacial: qué píxeles están cerca, dónde aparecen los bordes y cómo se relacionan las formas localmente. Los valores siguen en el vector, pero una capa Dense no aprovecha esa estructura de manera directa.

### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial de las imágenes?

**Respuesta:** Una red neuronal convolucional (CNN), porque sus filtros detectan patrones locales como bordes, texturas y formas, y combinan esos patrones en características más complejas.